Comprobar si hemos leido de Satg

In [2]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
print(orders.count(), items.count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
788 2010


Join de Order con Items

In [3]:
from pyspark.sql.functions import col

lines = items.join(orders, "order_id", "inner")
print(lines.count())

1980


Creamos una nueva columna con una formula

In [4]:
from pyspark.sql.functions import date_format

lines = (
    lines.withColumn(
        "gmv_line",
        col("qty") * col("unit_price") * (1 - col("discount")),
    ).withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
)
lines.select("order_id", "qty", "unit_price", "discount", "gmv_line", "order_month").show(5)
print("gmv nulos", lines.where(col("gmv_line").isNull()).count())
print("gmv < 0", lines.where(col("gmv_line") < 0).count())


+--------+---+----------+--------+--------+-----------+
|order_id|qty|unit_price|discount|gmv_line|order_month|
+--------+---+----------+--------+--------+-----------+
|  O00013|  2|    121.67|    0.00|243.3400|    2024-08|
|  O00013|  1|    128.00|    0.10|115.2000|    2024-08|
|  O00013|  5|    170.99|    0.10|769.4550|    2024-08|
|  O00013|  1|    237.56|    0.05|225.6820|    2024-08|
|  O00014|  2|    203.06|    0.00|406.1200|    2024-12|
+--------+---+----------+--------+--------+-----------+
only showing top 5 rows

gmv nulos 0
gmv < 0 13


In [12]:
lines.withColumn("is_high_value", col("gmv_line") >= 500).where("is_high_value").count()
# 506

506

In [13]:
lines2 = lines.withColumn("is_high_value", col("gmv_line") >= 500)

lines2.show()
# 506

+--------+----------+---+----------+--------+-----------+---------+-----------+-------------------+--------+-----------+-------------+
|order_id|product_id|qty|unit_price|discount|customer_id|   status|    channel|           order_ts|gmv_line|order_month|is_high_value|
+--------+----------+---+----------+--------+-----------+---------+-----------+-------------------+--------+-----------+-------------+
|  O00013|      P037|  2|    121.67|    0.00|      CX013|     paid|marketplace|2024-08-12 03:00:00|243.3400|    2024-08|        false|
|  O00013|      P026|  1|    128.00|    0.10|      CX013|     paid|marketplace|2024-08-12 03:00:00|115.2000|    2024-08|        false|
|  O00013|      P052|  5|    170.99|    0.10|      CX013|     paid|marketplace|2024-08-12 03:00:00|769.4550|    2024-08|         true|
|  O00013|      P010|  1|    237.56|    0.05|      CX013|     paid|marketplace|2024-08-12 03:00:00|225.6820|    2024-08|        false|
|  O00014|      P011|  2|    203.06|    0.00|      CX01